In [6]:
# use widgets as matplotlib backend
%matplotlib widget

# imports
from matplotlib import pyplot as plt
import os
import numpy
import os, sys
from time import time
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
import matplotlib as mpl

# import the file object for opening kongsberg files
# Note: function and library naming to be discussed
from themachinethatgoesping.echosounders import kongsbergall
from themachinethatgoesping.echosounders import index_functions
from themachinethatgoesping.pingprocessing import filter_pings

#simplify creating figures
mpl.rcParams['figure.dpi'] = 100
close_plots = True

def create_figure(name, return_ax=True):
    if close_plots: plt.close(name)
    fig = plt.figure(name)
    fig.suptitle = name

    if return_ax:
        return fig, fig.subplots()
    return fig

In [8]:
# list of folders with kongsberg .all or .wcd files (subfolders will be scanned as well)

folder = "../../../unittest_data/"

# list raw data files
files = index_functions.find_files(folder, [".all",".wcd"])
cacheFilePaths = index_functions.get_index_paths(file_paths=files)
files.sort()
            
files.sort()
file_name = files[0]
print("files:      ", len(files))

Found 16 files
files:       16


## Open all files
The function 
Notes: 
1. kongsbergall.KongsbergAllFileHandler(files) scanns and indexes all files and provides access to all files like a combined file stream
2. If a .all and a .wcd files with the same name (one .all and one .wcd) are added, they will be matched to a single file
3. It is not possible to add two .all or two .wcd with the same name, even if they are within different folders
4. Note: if the files are not sorted in time, the datagram packages will not be sorted by time either, however it isi simple to sort the pings at a later stage

In [9]:
# Index all files and initialize the file interfaces
# fm will be the accessor
fm = kongsbergall.KongsbergAllFileHandler(files)

#print some information about the files that where indexed
print(fm)

indexing files ⠐ 100% :00s<00m:00s] [..0871266855321420.all (1/16)]                               
indexing files ⠠ 100% :00s<00m:00s] [..3008643552583898.wcd (16/16)]                                
indexing files ⢀ 100% :00s<00m:00s] [Found: 712 datagrams in 16 files (10MB)]                                         
Initializing ping interface ⢀ 87% :00s<00m:00s] [Done]                                              
KongsbergAllFileHandler
#######################
-
File infos 
-------------               
- Number of loaded .all files: : 8        
- Number of loaded .wcd files: : 8        
- Total file size: :             10.71 MB 

 Detected datagrams 
^^^^^^^^^^^^^^^^^^^^ 
- timestamp_first:  21/08/2012 17:09:42.36 
- timestamp_last:   21/04/2023 17:48:15.35 
- Total:            712                    
- Datagrams [0x30]: 3                      [PUIDOutput]
- Datagrams [0x31]: 9                      [PUStatusOutput]
- Datagrams [0x33]: 3                      [ExtraParameters]
- Datag

## How to access ping data
1. Pings describe the main highlevel interface of Ping
2. They are easy to access using fm.get_pings
3. Pings combine information from the interfaces (e.g. navigation or configuration) with raw datagrams that where produced for each ping
4. Pings implement simple functions to access difficult data (e.g get_sv) However, this is still work in progress
5. Pings also provide access to the raw data, this will be used for plotting later

In [10]:
# Extract the 11th single ping for test
ping = fm.get_pings()[11]

# print the georeferenced position and attitude of the transducer
print(ping.get_geolocation())

GeolocationLatLon (struct)
##########################
- latitude:  38°24'4.2"S   [ddd°mm',ss.s''N/S]
- longitude: 142°29'1.6"E  [ddd°mm',ss.s''E/W]
- z:         0.600         [positive downwards, m]
- yaw:       13.546        [90 ° at east]
- pitch:     1.858         [° positive bow up]
- roll:      -1.492        [° positive port up]


In [11]:
# filter pings such that the for example must include water column and bottom detection data
pings = filter_pings.by_features(fm.get_pings(),['watercolumn','bottom'])
ping = pings[5]

In [12]:
# get access to the raw data of the specific ping
raw = ping.file_data

# print / access datagrams associated with the ping
print(raw.datagrams())

DatagramContainer
#################
-
Time info (Datagrams) 
------------------------ 
- Start time: 21/08/2012 17:09:43.39 
- End time:   21/08/2012 17:09:43.39 
- Sorted:     ascending              

 Contained datagrams 
--------------------- 
- Total:                           5 
- Datagrams [RawRangeAndAngle]:    1 [4e]
- Datagrams [XYZDatagram]:         1 [58]
- Datagrams [SeabedImageData]:     1 [59]
- Datagrams [WatercolumnDatagram]: 2 [6b]


In [13]:
#print the RawRangeAndAngle datagram associated with the ping
print(raw.datagrams("WatercolumnDatagram")[0])

WatercolumnDatagram
###################
- bytes:               63566    
- stx:                 0x02     
- datagram_identifier: 0x6b     [WatercolumnDatagram]
- model_number:        EM710    [710]
- date:                20120821 [YYYYMMDD]
- time_since_midnight: 61783394 [ms]

 date/time 
-----------  
- timestamp: 1345.569e⁶   [s]
- date:      21/08/2012   [MM/DD/YYYY]
- time:      17:09:43.394 [HH:MM:SS]

 datagram content 
------------------    
- ping_counter:                62490      
- system_serial_number:        210        
- number_of_datagrams:         2          
- datagram_number:             1          
- number_of_transmit_sectors:  3          
- total_no_of_receive_beams:   128        
- number_of_beams_in_datagram: 114        
- sound_speed:                 14658      [0.1 m/s]
- sampling_frequency:          231481     [0.01 Hz]
- tx_time_heave:               -102       [cm]
- tvg_function_applied:        30         
- tvg_offset_in_db:            0          
- scanni

In [14]:
# You may know that water column datagrams in the wcd/all files are split over multiple packtes.
# We therefore implemented a function that combines the split datagrams to a single one
wcd = raw.read_merged_watercolumndatagram()

print(wcd)

WatercolumnDatagram
###################
- bytes:               63566    
- stx:                 0x02     
- datagram_identifier: 0x6b     [WatercolumnDatagram]
- model_number:        EM710    [710]
- date:                20120821 [YYYYMMDD]
- time_since_midnight: 61783394 [ms]

 date/time 
-----------  
- timestamp: 1345.569e⁶   [s]
- date:      21/08/2012   [MM/DD/YYYY]
- time:      17:09:43.394 [HH:MM:SS]

 datagram content 
------------------    
- ping_counter:                62490      
- system_serial_number:        210        
- number_of_datagrams:         2          
- datagram_number:             1          
- number_of_transmit_sectors:  3          
- total_no_of_receive_beams:   128        
- number_of_beams_in_datagram: 128        
- sound_speed:                 14658      [0.1 m/s]
- sampling_frequency:          231481     [0.01 Hz]
- tx_time_heave:               -102       [cm]
- tvg_function_applied:        30         
- tvg_offset_in_db:            0          
- scanni

In [15]:
# read_merged_watercolumndatagram() holds a variable that is called skip_data
# if the true, the sample data will be skipped
# in this case you cannot access sample data, but looping through the data is accelerated 
# This is usefull to extract e.g statistical information in a first loop
# Speedup is not very impressive though (typically factor 2) because of the complicated datagram structure of .wcd files

# check if there where different tvgs applied in the datasets
tvgs = defaultdict(int)
for i,p in enumerate(tqdm(pings)):
    try:    
        w = p.file_data.read_merged_watercolumndatagram(skip_data = True)
        tvgs[w.get_tvg_function_applied()] += 1
              
    except IndexError as e:
        print("error:",i,e,"|",type(e))
    except ValueError as e:
        print("error:",i,e,"|",type(e))
    except RuntimeError as e:
        print("error:",i,e,"|",type(e))

for k,v in tvgs.items():
    print(f"TVG {k}: {v} pings")

  0%|          | 0/69 [00:00<?, ?it/s]

TVG 30: 63 pings
TVG 40: 6 pings
